# requirements

In [ ]:
import numpy as np
import pandas as pd

# 01.20.2026 debugging masked model

In [ ]:
# get_data() gives you a test input for the model

import os
from src.db_client import ClientCreator
import numpy as np
import pandas as pd
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader
import torch

def make_serial(samples_for_id, kind):
    return b"".join(
        [
            sample["chunk"]
            for sample in sorted(
                (
                    sample
                    for sample in samples_for_id
                    if sample["kind"] == kind
                ),
                key=lambda x: x["chunk_id"],
            )
        ]
    )

def multimodal_collate(results, field=("input", "modality", "label")):
    """
    Use this collate function with BatchPrefetchLoaderWrapper when using MultimodalMongoDataset.
    It will stack the inputs, modality codes, and labels into tensors properly.
    """
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }

    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.float()

def get_data(batch = list(range(1,11))):
    host = "10.245.12.58"
    host_slurm = "arctrdcn018.rs.gsu.edu"
    db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
    crop_tensor = False
    client_creator = ClientCreator(
        db_host, crop_tensor=crop_tensor
    )

    db_name = databases = "MindfulTensors"
    db_collection = collections = "fbirn"
    meta_fields = ["gender_encoded"]
    numcubes = 1
    cubesizes = 256
    subvolume_shape = [cubesizes] * 3

    client_creator.set_database(databases)
    client_creator.set_collection(collections)
    client_creator.set_num_subcubes(numcubes)
    client_creator.set_shape(subvolume_shape)

    funcs = {
        "createclient": client_creator.create_client,
        "createVclient": client_creator.create_client,
        "mycollate": client_creator.mycollate,
        "mycollate_full": client_creator.mycollate_full,
        "mytransform": client_creator.mytransform,
    }

    db_fields = ["falff", "smri", "dwi"]

    client = MongoClient("mongodb://" + db_host + ":27017")
    db = client[db_name]
    posts_bin = db[db_collection + ".bin"]
    posts_meta = db[db_collection + ".meta"]



    fields = {"id": 1, "chunk": 1, "kind": 1, "chunk_id": 1}
    samples = list(
        posts_bin.find(
            {
                "id": {"$in": batch},
                "kind": {"$in": db_fields}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
            },
            fields,
        )
    )

    results = {}

    for id in batch:
        # get ID's label and modalities
        meta_for_id = list(
            posts_meta.find(
                {
                    "id": id,
                },
                meta_fields + ["modalities"],
            )
        )

        assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
        assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"

        label = meta_for_id[0][meta_fields[0]]
        modalities = meta_for_id[0]["modalities"]
        id_modalities = set(modalities).intersection(set(db_fields))

        # Get samples for this ID
        samples_for_id = [
            sample
            for sample in samples
            if sample["id"] == id
        ]

        for mod in id_modalities:
            data = make_serial(samples_for_id, mod)

            for mod in id_modalities:
                data = make_serial(samples_for_id, mod)

                result = {
                    "input": unit_interval_normalize(funcs["mytransform"](data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }

                results[str(id)+'_'+mod] = result

    return multimodal_collate(results)

In [ ]:
data, modalities, labels = get_data(batch=[1,2,4,7])

data.shape, modalities.shape, labels.shape

In [ ]:
from resnet import ResNet3D, BasicBlock3D
from torch import nn
from torch.nn import functional as F
import torch

from collections import OrderedDict
from copy import deepcopy
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1, # LR is not required for calculating scores [so far]
                                    momentum=0.9,
                                    weight_decay=1e-4)
        
        self.model_device = next(iter(model.parameters())).device

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        Generates and stores masks for each modality found in the input.
        """
        self.masks_storage = {}
        unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        
        for mod in unique_modalities:
            # 1. Filter data for this modality
            mask = (modalities == mod)
            mod_data = input_data[mask]
            mod_labels = labels[mask]
            
            # 2. Generate SNIP masks
            batch = (mod_data, mod_labels)
            print(f"Generating masks for modality: {mod}")
            
            # Note: generate_mask_from_grad_scores returns (masks_by_name, masks_by_layer)
            # We only need masks_by_name for storage
            masks_by_name, _ = self.generate_mask_from_grad_scores(batch)
            
            # 3. Store in our dictionary
            self.masks_storage[mod] = masks_by_name
            
        print(f"Masks registered for modalities: {list(self.masks_storage.keys())}")

    def forward(self, input_data, modalities):
        """
        The Manual Forward Strategy:
        1. Split input by modality.
        2. Swap weights with masked weights.
        3. Run forward.
        4. Restore weights.
        """
        # Prepare an output tensor (assuming binary classification with 1 output)
        batch_size = input_data.shape[0]
        final_outputs = torch.zeros(batch_size, 1, device=self.model_device) 
        
        unique_mods = torch.unique(modalities).cpu().detach().tolist()

        for mod in unique_mods:
            # A. Identify data for this modality
            idx_map = (modalities == mod)
            sub_data = input_data[idx_map]
            
            if sub_data.shape[0] == 0:
                continue

            # B. Apply the mask for this modality (Temporarily overwrite weights)
            # We assume register_multimodal_masks has been called already
            if mod in self.masks_storage:
                self._apply_mask_to_model(mod)
            else:
                print(f"Warning: No mask found for modality {mod}, using full weights.")

            # C. Run the forward pass
            # The model now thinks it has pruned weights
            sub_output = self.model(sub_data)
            
            # D. Restore original weights (CRITICAL STEP)
            self._restore_model_weights()
            
            # E. Store results in the final tensor
            final_outputs[idx_map] = sub_output

        return final_outputs

    def _apply_mask_to_model(self, mod_id):
        """
        Temporarily replaces layer.weight with layer.weight * mask.
        Saves the original parameter reference to restore later.
        """
        masks = self.masks_storage[mod_id]
        
        # We need to save the original parameters so we can put them back
        self.saved_parameters = {} 

        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS) and name in masks:
                # 1. Save the actual Parameter object (which the optimizer tracks)
                self.saved_parameters[name] = module.weight
                
                # 2. Get the mask
                mask = masks[name]
                
                # 3. Overwrite module.weight with a Tensor (not a Parameter)
                # This keeps the gradient graph alive: (Weight * Mask) -> Output
                # NOT using .data here ensures gradients flow back to module.weight
                module.weight = module.weight * mask

    def _restore_model_weights(self):
        """
        Puts the original Parameter objects back into the layers.
        """
        for name, module in self.model.named_modules():
            if name in self.saved_parameters:
                # Restore the reference to the original Parameter
                module.weight = self.saved_parameters[name]
        
        # Clear the temp storage
        self.saved_parameters = {}

    ### mask generation section
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = dict()
        for name, values in scores_dict.items():
            masks[name] = ((values) >= threshold).float().to(self.model_device)

        mask_key_layer = dict()
        # we have to iterate over the original model which we wish to mask, not the copied
        # cpu model, because we are using the exact layers of that model as keys to the mask
        # dictionary.
        for name, layer in self.model.named_modules(): 
            if isinstance(layer, PRUNE_LAYERS):
                mask_key_layer[layer] = masks[name]
        return masks, mask_key_layer
    
    def _calculate_scores(self, batch):
        """
        Calculate the gradient scores for later thresholding and masking.
        """
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        # compute output
        preds = self.cpu_model(data)
        # print(f"Preds shape: {preds.shape}, \n Preds: {preds} \n labels: {labels}")
        loss = F.binary_cross_entropy_with_logits(preds, labels)
        # print(f"Loss: {loss}")
        # compute gradient and do SGD step
        self.cpu_optimizer.zero_grad()
        loss.backward()
        # Calculate the score from the weights and the gradients
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # print(f"Calculating scores for layer: {name} with weight shape: {module.weight.shape}")
                # print(f"Weight min value: {module.weight.data.min()}, max value: {module.weight.data.max()}")
                # print(f"Grad min value: {module.weight.grad.min()}, max value: {module.weight.grad.max()}")
                scores_d[name] = (module.weight.grad * module.weight.data.detach()).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        """
        Calculate the threshold value from the gradient scores for masking.
        TODO: ask Riyasat if it won't lead to zeroing out some layer by accident. 
        It prolly can, but the chances are low.
        """
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        # not normalizing here, since it doesn't look like it's needed. Look here again if performance suffers.
        num_params_to_keep = int(len(global_scores) * (1.0-self.sparsity))
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        threshold = topk_scores[-1]

        # print(f"Global scores stats: Max={global_scores.max()}, Min={global_scores.min()}, Mean={global_scores.mean()}")
        # print(f"Calculated Threshold: {threshold}")
        # print(f"Targeting to keep: {num_params_to_keep} out of {len(global_scores)}")

        return threshold

    ### hooks

    def _unregister_mask(self):
        for module in self.model.modules():
            module._backward_hooks = OrderedDict()
            module._forward_pre_hooks = OrderedDict()


    @staticmethod
    def _forward_pre_hook(module, x):
        module.mask.requires_grad_(False)
        mask = module.mask
        module.weight.data.mul_(mask.to(module.weight.get_device())) # This might be a place of performance loss because to -> .to()



In [ ]:
from resnet import ResNet3D

model_channels = 64 # from config
n_classes = 1 # from config, binary

resnet_model = ResNet3D(
        in_channels=1, 
        n_classes=n_classes, 
        channels=model_channels,
    )

model = MultiMaskSNIPWrapper(resnet_model)

In [ ]:
# model.register_multimodal_masks(modalities, data, labels)
model.register_multimodal_masks(modalities, data, labels)

In [ ]:
test = model(data, modalities)

In [ ]:
# CHECK THAT GRADIENTS MATCH THE MASKS 

In [ ]:
# masks.keys()
def get_tensor_sps(tensor):
    nz_count = torch.count_nonzero(tensor).item()
    total_params = tensor.numel()

    return nz_count/total_params

for key in model.masks[list(model.masks.keys())[0]]:
    print(f"Layer: {key}, Sparsity: {100*(1 - get_tensor_sps(model.masks[list(model.masks.keys())[0]][key])):.2f} %")

In [ ]:
# backup

from resnet import ResNet3D, BasicBlock3D
from torch import nn
from torch.nn import functional as F
import torch

from collections import OrderedDict
from copy import deepcopy
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1, # LR is not required for calculating scores [so far]
                                    momentum=0.9,
                                    weight_decay=1e-4)
        
        self.model_device = next(iter(model.parameters())).device

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        The idea if this function is to be used as the MASKs initialization step.
        Pass modalities, pass input data and labels.
        The function will compute masks based on gradients for each modality and then register them.
        """

        self.masks = {}

        # derive the masks for each modality
        # unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        unique_modalities = torch.unique(modalities)
        for mod in unique_modalities:
            mod_data = input_data[modalities == mod]
            mod_labels = labels[modalities == mod]
            batch = (mod_data, mod_labels)
            print(f"Generating masks for modality: {mod} with data shape: {mod_data.shape}, labels shape: {mod_labels.shape}, unique labels: {torch.unique(mod_labels)}")
            masks, _ = self.generate_mask_from_grad_scores(batch)
            
            self.masks[mod] = masks

        # register masks in module buffers
        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                device = module.weight.get_device()
                device = 'cpu' if device == -1 else f"cuda:{device}"
                module.masks = {mod: nn.Parameter(self.masks[mod][name].requires_grad_(False).to(device)) for mod in self.masks.keys()}
                # module.masks = nn.Parameter(masks[module]).requires_grad_(False).to(module.weight.get_device())
                # hook registration is performed dynamically in the forward for each modality
                # module.register_forward_pre_hook(self._forward_pre_hook)
                

    ### mask generation section
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = dict()
        for name, values in scores_dict.items():
            masks[name] = ((values) >= threshold).float().to(self.model_device)

        mask_key_layer = dict()
        # we have to iterate over the original model which we wish to mask, not the copied
        # cpu model, because we are using the exact layers of that model as keys to the mask
        # dictionary.
        for name, layer in self.model.named_modules(): 
            if isinstance(layer, PRUNE_LAYERS):
                mask_key_layer[layer] = masks[name]
        return masks, mask_key_layer
    
    def _calculate_scores(self, batch):
        """
        Calculate the gradient scores for later thresholding and masking.
        """
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        # compute output
        preds = self.cpu_model(data)
        # print(f"Preds shape: {preds.shape}, \n Preds: {preds} \n labels: {labels}")
        loss = F.binary_cross_entropy_with_logits(preds, labels)
        # print(f"Loss: {loss}")
        # compute gradient and do SGD step
        self.cpu_optimizer.zero_grad()
        loss.backward()
        # Calculate the score from the weights and the gradients
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # print(f"Calculating scores for layer: {name} with weight shape: {module.weight.shape}")
                # print(f"Weight min value: {module.weight.data.min()}, max value: {module.weight.data.max()}")
                # print(f"Grad min value: {module.weight.grad.min()}, max value: {module.weight.grad.max()}")
                scores_d[name] = (module.weight.grad * module.weight.data.detach()).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        """
        Calculate the threshold value from the gradient scores for masking.
        TODO: ask Riyasat if it won't lead to zeroing out some layer by accident. 
        It prolly can, but the chances are low.
        """
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        # not normalizing here, since it doesn't look like it's needed. Look here again if performance suffers.
        num_params_to_keep = int(len(global_scores) * (1.0-self.sparsity))
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        threshold = topk_scores[-1]

        # print(f"Global scores stats: Max={global_scores.max()}, Min={global_scores.min()}, Mean={global_scores.mean()}")
        # print(f"Calculated Threshold: {threshold}")
        # print(f"Targeting to keep: {num_params_to_keep} out of {len(global_scores)}")

        return threshold

    ### hooks

    def _unregister_mask(self):
        for module in self.model.modules():
            module._backward_hooks = OrderedDict()
            module._forward_pre_hooks = OrderedDict()


    @staticmethod
    def _forward_pre_hook(module, x):
        module.mask.requires_grad_(False)
        mask = module.mask
        module.weight.data.mul_(mask.to(module.weight.get_device())) # This might be a place of performance loss because to -> .to()


    # def get_model_sps(self):
    #     nonzero = total = 0
    #     for name, param in self.model.named_parameters():
    #         if 'mask' not in name:
    #             tensor = param.detach().clone()
    #             # nz_count.append(torch.count_nonzero(tensor))
    #             nz_count = torch.count_nonzero(tensor).item()
    #             total_params = tensor.numel()
    #             nonzero += nz_count
    #             total += total_params
        
    #     tensor = None
    #     abs_sps = 100 * (total-nonzero) / total
    #     return abs_sps
    
    # def print_model_sps(self):
    #     print(f"Model Sparsity: {self.get_model_sps():.2f} %")

    # def forward(self, x):
    #     mask = self.masks.iloc[self.current_mask_index].values
    #     mask_tensor = {name: torch.tensor(mask_value, dtype=torch.float32) 
    #                    for name, mask_value in zip(self.model.state_dict().keys(), mask)}
        
    #     for name, param in self.model.named_parameters():
    #         if name in mask_tensor:
    #             param.data *= mask_tensor[name]
        
    #     return self.model(x)
    

# 01.12.2026 updating for mutlimodal training

In [ ]:
import os
from src.db_client import ClientCreator

host = "10.245.12.58"
host_slurm = "arctrdcn018.rs.gsu.edu"
db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
crop_tensor = False
client_creator = ClientCreator(
    db_host, crop_tensor=crop_tensor
)

db_name = databases = "MindfulTensors"
db_collection = collections = "fbirn"
meta_fields = ["gender_encoded"]
index_id = 'id'
numcubes = 1
cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]
numvolumes = num_volumes = 4
prefetches = 16

client_creator.set_database(databases)
client_creator.set_collection(collections)
client_creator.set_num_subcubes(numcubes)
client_creator.set_shape(subvolume_shape)

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}


In [ ]:
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader

db_fields = ["falff"]
db_fields = ["smri"]
db_fields = ["dwi"]
db_fields = ["falff", "smri", "dwi"]

client = MongoClient("mongodb://" + db_host + ":27017")
db = client[db_name]
posts_bin = db[db_collection + ".bin"]
posts_meta = db[db_collection + ".meta"]

all_ids = posts_meta.distinct( # pull all unique IDs (subjects) with at least one modality in db_fields
    "id",
    {'modalities': {"$in": db_fields}}
)
all_ids = sorted(all_ids)
# print(all_ids)

labels = []
for id in all_ids:
    label = posts_meta.find_one({"id": id}, meta_fields)[meta_fields[0]] # get label for the id
    labels.append(label)
labels = np.array(labels)
labels.shape, len(all_ids)

In [ ]:
# checking the samples, need to figure out how to stitch modalities together correctly

batch = [1, 2, 3, 4]

metas = list(posts_meta.find({
    "id": {"$in": batch}
}))

bins = list(posts_bin.find(
    {
    "id": {"$in": batch}
    },
))

# len(metas), len(bins), metas[1]
bins[0].keys(), metas[0].keys(), metas[0]["modalities"]

In [ ]:
from mindfultensors.mongoloader import MongoDataset
import torch
from mindfultensors.utils import unit_interval_normalize
from torch.utils.data import TensorDataset

class MultimodalMongoDataset(MongoDataset):
    def __init__(
        self,
        indices,
        transform,
        collection,
        sample,
        meta_sample,
        normalize=unit_interval_normalize,
        id="id",
    ):
        super(MultimodalMongoDataset, self).__init__(indices, transform, collection, sample, normalize, id)
        self.meta_sample = meta_sample

    def __getitem__(self, batch):
        # Fetch all samples for ids in the batch and where 'kind' is either
        # data or label as specified by the sample parameter
        # TODO: make it respect the batch size; right now it returns a bigger batch with multiple modalities per id
        
        samples = list(
            self.collection["bin"].find(
                {
                    self.id: {"$in": [self.indices[_] for _ in batch]},
                    "kind": {"$in": self.sample}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
                },
                self.fields,
            )
        )

        results = {}
        for id in batch:
            # get ID's label and modalities
            meta_for_id = list(
                self.collection["meta"].find(
                    {
                        self.id: self.indices[id],
                    },
                    self.meta_sample + ["modalities"],
                )
            )

            assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
            assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"
            
            label = meta_for_id[0][self.meta_sample[0]]
            modalities = meta_for_id[0]["modalities"]
            id_modalities = set(modalities).intersection(set(self.sample))
                

            # Get samples for this ID
            samples_for_id = [
                sample
                for sample in samples
                if sample[self.id] == self.indices[id]
            ]

            for mod in id_modalities:
                data = self.make_serial(samples_for_id, mod)

                result = {
                    "input": self.normalize(self.transform(data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }


                # Add to results
                results[str(id)+'_'+mod] = result

        return results

In [ ]:
# info on collate mystery 

crop_tensor = False

cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]

def mycollate_full(self, x):
    return crop_tensor(*mcollate(x)) if self.crop_tensor else mcollate(x)

def mcollate(results, field=("input", "label")):
    results = results[0]
    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    label_tensors = [results[id_][field[1]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_labels.long()

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}

collate = (
    funcs["mycollate_full"]
    if shape == 256
    else funcs["mycollate"]
)

In [ ]:
from src.customMongoDataset import CustomMongoDataset

# train_dataset = CustomMongoDataset(
train_dataset = MultimodalMongoDataset(
    all_ids, 
    funcs["mytransform"],
    None,
    db_fields,
    meta_fields,
    normalize=unit_interval_normalize,
    id=index_id,
)

def multimodal_collate(results, field=("input", "modality", "label")):
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }
    results = results[0]
    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.long()

collate = multimodal_collate

train_sampler = DBBatchSampler(train_dataset, batch_size=num_volumes)
train_dataloader = BatchPrefetchLoaderWrapper(
    DataLoader(
        train_dataset,
        sampler=train_sampler,
        collate_fn=collate,
        pin_memory=True,
        worker_init_fn=funcs["createclient"],
        persistent_workers=True,
        prefetch_factor=1,
        num_workers=1,  # prefetches,
        # prefetch_factor=None,
        # num_workers=1,  # prefetches,
    ),
    num_prefetches=prefetches,
)

In [ ]:
for batch in train_dataloader:
    print(len(batch))
    print(batch[0].shape)
    print(batch[1])
    break